# 🚀 Automated Algorithmic Options Trading System
**Features:**
- Multi-Key Angel One API Connection
- Real-time Greeks Calculation (Delta, Gamma, Theta, Vega, Rho)
- Automated Strategy Execution
- Risk Management & Budget Control (10k/20k Hard Limit)
- Live Dashboard

In [ ]:
# --- CELL 1: IMPORTS & SETUP ---
import os
import time
import datetime
import pandas as pd
import numpy as np
import pyotp
import requests
import json
import logging
from SmartApi import SmartConnect
from py_vollib.black_scholes import black_scholes
from py_vollib.black_scholes.greeks.analytical import delta, gamma, rho, theta, vega
import threading
from IPython.display import display, clear_output

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

print("✅ Libraries Imported Successfully!")

In [ ]:
# --- CELL 2: ANGEL ONE MULTI-KEY MANAGER ---

# ==========================================
# 1. CREDENTIALS POOL
# ==========================================
CREDENTIALS_POOL = [
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
    {"api_key": "***REDACTED***", "client": "***REDACTED***", "pwd": "1234", "totp": "***REDACTED***"},
]

class AngelKeyManager:
    def __init__(self, creds_list):
        self.sessions = []
        self.current_index = 0
        print(f"Initializing {len(creds_list)} API Sessions...")
        
        for cred in creds_list:
            try:
                obj = SmartConnect(api_key=cred['api_key'])
                totp = pyotp.TOTP(cred['totp']).now()
                data = obj.generateSession(cred['client'], cred['pwd'], totp)
                if data['status']:
                    self.sessions.append(obj)
                    print(f" >> Session Active for Key: ...{cred['api_key'][-4:]}")
                else:
                    print(f" >> Failed for Key: ...{cred['api_key'][-4:]} | {data['message']}")
            except Exception as e:
                print(f" >> Error: {e}")
        
        if not self.sessions:
            raise Exception("No API Sessions could be established!")

    def get_session(self):
        """Returns a session and rotates the index"""
        session = self.sessions[self.current_index]
        self.current_index = (self.current_index + 1) % len(self.sessions)
        return session

# Initialize Manager
key_manager = AngelKeyManager(CREDENTIALS_POOL)
print("✅ API Manager Ready!")

In [ ]:
# --- CELL 3: GREEKS CALCULATOR (QUANT ENGINE) ---

class GreeksCalculator:
    @staticmethod
    def calculate_greeks(S, K, T, r, sigma, option_type='c'):
        """
        S: Spot Price
        K: Strike Price
        T: Time to Expiration (in years)
        r: Risk-free Interest Rate (decimal, e.g., 0.07)
        sigma: Volatility (decimal, e.g., 0.20)
        option_type: 'c' for Call, 'p' for Put
        """
        try:
            # Handle edge case where T is 0 (Expiration Day)
            if T <= 0:
                return {"delta": 0, "gamma": 0, "theta": 0, "vega": 0, "rho": 0, "price": 0}

            price = black_scholes(option_type, S, K, T, r, sigma)
            d = delta(option_type, S, K, T, r, sigma)
            g = gamma(option_type, S, K, T, r, sigma)
            t = theta(option_type, S, K, T, r, sigma)
            v = vega(option_type, S, K, T, r, sigma)
            rh = rho(option_type, S, K, T, r, sigma)
            
            return {
                "price": round(price, 2),
                "delta": round(d, 4),
                "gamma": round(g, 4),
                "theta": round(t, 4),
                "vega": round(v, 4),
                "rho": round(rh, 4)
            }
        except Exception as e:
            # print(f"Greeks Error: {e}")
            return {"delta": 0, "gamma": 0, "theta": 0, "vega": 0, "rho": 0, "price": 0}

# Test Calculator
test_greeks = GreeksCalculator.calculate_greeks(S=24000, K=24100, T=5/365, r=0.07, sigma=0.15, option_type='c')
print(f"Test Greeks (NIFTY 24000 Spot, 24100 Call, 5 Days left): {test_greeks}")

In [ ]:
# --- CELL 4: RISK & BUDGET MANAGER ---

class RiskManager:
    def __init__(self, total_budget=20000, max_risk_per_trade=0.02):
        self.total_budget = total_budget
        self.current_capital = total_budget
        self.max_risk_per_trade = max_risk_per_trade # 2% risk
        self.active_positions = []
        
    def check_entry(self, price, lot_size=25):
        """
        Determines if we can take the trade and how many lots.
        """
        cost_per_lot = price * lot_size
        
        # 1. Budget Check
        if cost_per_lot > self.current_capital:
            print(f"❌ Rejected: Cost {cost_per_lot} > Capital {self.current_capital}")
            return 0
        
        # 2. Position Sizing (Simple: Max 1 lot for now to be safe, or calculate based on risk)
        # For this system, we'll stick to 1 lot per signal to start.
        return 1
    
    def update_capital(self, pnl):
        self.current_capital += pnl
        print(f"💰 Capital Updated: {self.current_capital}")

risk_manager = RiskManager(total_budget=20000)
print(f"✅ Risk Manager Initialized. Budget: ₹{risk_manager.total_budget}")

In [ ]:
# --- CELL 5: DATA FETCHER (SPOT & OPTIONS) ---

class DataFetcher:
    def __init__(self, key_manager):
        self.key_manager = key_manager
        self.tokens = {
            "NIFTY": {"token": "99926000", "symbol": "Nifty 50", "exch_seg": "NSE"},
            "BANKNIFTY": {"token": "99926009", "symbol": "Nifty Bank", "exch_seg": "NSE"}
        }
    
    def get_spot_price(self, symbol="NIFTY"):
        session = self.key_manager.get_session()
        token_info = self.tokens.get(symbol)
        
        try:
            # Fetch LTP
            data = session.ltpData(token_info['exch_seg'], token_info['symbol'], token_info['token'])
            if data['status']:
                return data['data']['ltp']
        except Exception as e:
            print(f"Error fetching Spot: {e}")
        return None

    def get_option_chain(self, spot_price, symbol="NIFTY", expiry_date="28NOV2024"):
        """
        Generates a list of ATM, ITM, OTM strikes based on spot price.
        Note: In a real scenario, we would fetch the master contract list to get exact tokens.
        For this demo, we will simulate the selection logic.
        """
        step = 50 if symbol == "NIFTY" else 100
        atm_strike = round(spot_price / step) * step
        
        strikes = [
            atm_strike - step, # ITM Call / OTM Put
            atm_strike,        # ATM
            atm_strike + step  # OTM Call / ITM Put
        ]
        
        return strikes

data_fetcher = DataFetcher(key_manager)
# Test Spot
spot = data_fetcher.get_spot_price("NIFTY")
print(f"Current NIFTY Spot: {spot}")

In [ ]:
# --- CELL 6: STRATEGY ENGINE ---

class StrategyEngine:
    def __init__(self):
        pass
    
    def analyze(self, market_data):
        """
        market_data: { 'spot': 24000, 'atm_call_greeks': {...}, 'atm_put_greeks': {...} }
        Returns: 'BUY_CALL', 'BUY_PUT', or 'HOLD'
        """
        # 1. Simple Momentum Strategy
        # If Call Delta > 0.5 and Gamma is rising -> Buy Call
        call_greeks = market_data.get('atm_call_greeks')
        put_greeks = market_data.get('atm_put_greeks')
        
        if call_greeks and call_greeks['delta'] > 0.55:
            return "BUY_CALL"
        
        if put_greeks and put_greeks['delta'] < -0.55:
            return "BUY_PUT"
            
        return "HOLD"

strategy = StrategyEngine()
print("✅ Strategy Engine Loaded")

In [ ]:
# --- CELL 7: LIVE TRADING LOOP ---

def run_live_trading():
    print("🚀 STARTING LIVE TRADING SYSTEM...")
    print(f"💰 Initial Budget: {risk_manager.total_budget}")
    
    try:
        while True:
            # 1. Fetch Spot
            spot = data_fetcher.get_spot_price("NIFTY")
            if not spot:
                print("⚠️ Failed to fetch Spot. Retrying...")
                time.sleep(2)
                continue
                
            # 2. Simulate Option Data (Since we don't have full master list loaded yet)
            # In real impl, we fetch LTP for the ATM strike
            strikes = data_fetcher.get_option_chain(spot)
            atm_strike = strikes[1]
            
            # 3. Calculate Greeks (Mocking Volatility as 15%)
            days_to_expiry = 5 # Mock
            call_greeks = GreeksCalculator.calculate_greeks(spot, atm_strike, days_to_expiry/365, 0.07, 0.15, 'c')
            put_greeks = GreeksCalculator.calculate_greeks(spot, atm_strike, days_to_expiry/365, 0.07, 0.15, 'p')
            
            # 4. Strategy Analysis
            market_data = {
                'spot': spot,
                'atm_call_greeks': call_greeks,
                'atm_put_greeks': put_greeks
            }
            signal = strategy.analyze(market_data)
            
            # 5. Dashboard Update
            clear_output(wait=True)
            print(f"🔴 LIVE MARKET | {datetime.datetime.now().strftime('%H:%M:%S')}")
            print(f"--------------------------------------------------")
            print(f"📈 NIFTY SPOT : {spot}")
            print(f"🎯 ATM STRIKE : {atm_strike}")
            print(f"--------------------------------------------------")
            print(f"🟢 CALL GREEKS: Delta: {call_greeks['delta']} | Theta: {call_greeks['theta']}")
            print(f"🔴 PUT GREEKS : Delta: {put_greeks['delta']} | Theta: {put_greeks['theta']}")
            print(f"--------------------------------------------------")
            print(f"🤖 SIGNAL     : {signal}")
            print(f"💰 CAPITAL    : {risk_manager.current_capital}")
            
            # 6. Execution (Mock)
            if signal == "BUY_CALL":
                lots = risk_manager.check_entry(call_greeks['price'])
                if lots > 0:
                    print(f"⚡ EXECUTING BUY CALL {atm_strike} | {lots} Lots")
                    # Mock Trade
                    risk_manager.current_capital -= (call_greeks['price'] * 25 * lots)
            elif signal == "BUY_PUT":
                lots = risk_manager.check_entry(put_greeks['price'])
                if lots > 0:
                    print(f"⚡ EXECUTING BUY PUT {atm_strike} | {lots} Lots")
                    # Mock Trade
                    risk_manager.current_capital -= (put_greeks['price'] * 25 * lots)
            
            time.sleep(5)
            
    except KeyboardInterrupt:
        print("🛑 Trading Stopped by User")

# Uncomment to run
# run_live_trading()